# Team Assigner -- SigLIP clustering on STABLE ids (sampled, no full-match pass)

Same method as the windowed notebook (SigLIP embedding of the torso crop -> StandardScaler -> KMeans(k=2) -> vote per id,
goalkeepers by pitch-space centroid proximity, referees excluded, kit colours measured from the real torso pixels), but
built for the new tracking where **every id is one person for the whole match**:

* **No full-match classification pass.** A fixed number of crops per id (`SAMPLES_PER_ID`, spread over the whole match,
  preferring large boxes with no other box touching them) is enough, because the id never changes person. ~1,200
  SigLIP calls instead of ~120,000, and the calibration and classification use the same crops.
* **No windows.** One vote per id over its samples; the vote share is the confidence (`WEAK_MAJORITY_THRESHOLD`).
* **One video pass**, sequential (frame-accurate crops), only decoding the frames that are needed.
* Team labels do **not** use names. `player_info` (if present) is only used to print shirt numbers.

Outputs: the **normal structure** (same keys as before, so `frame_table.py`, `passes`, pressure and the renderer keep
working) and the **simple structure** (`team_by_id`, `vote_share_by_id`, `team_colors`).

**General version (works on any stable-id cache, no names used):** teams are found from the kits only (team 0/1 is
arbitrary). Crops are taken only from real boxes: interpolated, duplicate and auto-assigned boxes are skipped when the
cache marks them (caches without these flags work unchanged). People of class `other` (off-pitch / extra) are ignored.
Goalkeepers get their team by a **per-frame vote** (nearest team centroid in pitch space in each sampled frame), so it
also works over a full match with the half-time side swap.

## Cell 1 -- Config

In [ ]:
import os
import cv2
import pickle
import colorsys
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from tqdm.auto import tqdm

TRACKING_CACHE_PATH   = "/kaggle/working/barca_atletico_tracking_cache_named.pkl"   # stable-id tracking cache
VIDEO_PATH            = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv"
HOMOGRAPHY_CACHE_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/homography_cache_barca_atletico_firsthalf.pkl"

OUTPUT_TEAM_CACHE_PATH  = "/kaggle/working/barca_atletico_team_assignment_stable.pkl"          # normal structure
OUTPUT_TEAM_SIMPLE_PATH = "/kaggle/working/barca_atletico_team_assignment_stable_simple.pkl"   # simple structure
OUTPUT_VIDEO_PATH       = "/kaggle/working/barca_atletico_team_annotated_stable.mp4"

# ---- sampling (per id, spread over the match) ----
SAMPLES_PER_ID = 60          # crops per player id (60 x 20 players = 1,200 SigLIP calls)
CANDIDATE_STRIDE = 5         # only frames f % 5 == 0 are candidates (cheaper isolation check)
ISO_EXPAND = 0.10            # a box touching another box enlarged by 10% is not "isolated" (occlusion)

# ---- torso crop (same as before) ----
TORSO_TOP_RATIO = 0.15
TORSO_BOTTOM_RATIO = 0.50
TORSO_SIDE_MARGIN = 0.20
MIN_BBOX_AREA = 900

# ---- votes / colours ----
WEAK_MAJORITY_THRESHOLD = 0.7
GRASS_DOMINANCE_THRESH = 1.15

# ---- pitch projection (goalkeepers) ----
PX_PER_METER = 10
POSITION_SAMPLE_STRIDE = 10

# ---- SigLIP ----
SIGLIP_MODEL_NAME = "google/siglip-base-patch16-224"
EMBED_BATCH = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- which boxes may be cropped (flags written by the naming notebook; absent flags = real box) ----
SKIP_FOR_CROPS = ("interpolated", "duplicate", "auto_assigned")
GK_MIN_PLAYERS_PER_TEAM = 3   # a frame votes for a goalkeeper only if both teams have >= 3 players on screen

RENDER_VIDEO = False         # True = write the annotated check video (full-match render, slow)

print("Checking paths...")
for p in [TRACKING_CACHE_PATH, VIDEO_PATH, HOMOGRAPHY_CACHE_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")
print(f"Device: {DEVICE}")


## Cell 2 -- Load tracking cache + homography cache

In [ ]:
with open(TRACKING_CACHE_PATH, "rb") as f:
    saved = pickle.load(f)
if isinstance(saved, dict) and "tracking_cache" in saved:
    tracking_cache = saved["tracking_cache"]
    locked_class_by_id = saved["locked_class_by_id"]
    player_info = saved.get("player_info", {})        # display only (shirt numbers), never used for teams
else:
    tracking_cache, player_info = saved, {}
    votes = defaultdict(Counter)
    for d in tracking_cache.values():
        for t in d["tracks"]:
            votes[t["track_id"]][t["class"]] += 1
    locked_class_by_id = {tid: v.most_common(1)[0][0] for tid, v in votes.items()}

with open(HOMOGRAPHY_CACHE_PATH, "rb") as f:
    homography_cache = pickle.load(f)

def get_homography(frame_idx):
    if isinstance(homography_cache, dict):
        return homography_cache.get(frame_idx)
    return homography_cache[frame_idx] if 0 <= frame_idx < len(homography_cache) else None

def show_id(tid):
    num = player_info.get(tid, {}).get("number")
    return f"{tid}" if num is None else f"{tid} (#{num})"

player_ids = sorted(tid for tid, c in locked_class_by_id.items() if c == "player")
goalkeeper_ids = sorted(tid for tid, c in locked_class_by_id.items() if c == "goalkeeper")
referee_ids = sorted(tid for tid, c in locked_class_by_id.items() if c == "referee")
_other = sorted(tid for tid, c in locked_class_by_id.items() if c not in ("player", "goalkeeper", "referee"))
if _other:
    print(f"Ignored (not player / goalkeeper / referee, e.g. off-pitch people): {len(_other)} ids")
det_by_id = Counter(t["track_id"] for d in tracking_cache.values() for t in d["tracks"])
print(f"Tracking cache: {len(tracking_cache)} frames")
print(f"Players: {len(player_ids)} | goalkeepers: {goalkeeper_ids} | referees: {referee_ids}")
if len(player_ids) > 40:
    print("  [warn] many player ids -> this cache is probably NOT the stable-id one (use the windowed notebook for it)")


## Cell 3 -- SigLIP model + torso crop + batched embedding

In [ ]:
from transformers import SiglipVisionModel, SiglipImageProcessor
from PIL import Image

siglip_processor = SiglipImageProcessor.from_pretrained(SIGLIP_MODEL_NAME)
siglip_model = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_NAME).to(DEVICE).eval()


def get_torso_crop(frame, bbox):
    x1, y1, x2, y2 = bbox
    h_img, w_img = frame.shape[:2]
    bw, bh = (x2 - x1), (y2 - y1)
    if bw <= 0 or bh <= 0 or bw * bh < MIN_BBOX_AREA:
        return None
    x1c, y1c = int(round(x1 + TORSO_SIDE_MARGIN * bw)), int(round(y1 + TORSO_TOP_RATIO * bh))
    x2c, y2c = int(round(x2 - TORSO_SIDE_MARGIN * bw)), int(round(y1 + TORSO_BOTTOM_RATIO * bh))
    x1c, y1c, x2c, y2c = max(0, x1c), max(0, y1c), min(w_img, x2c), min(h_img, y2c)
    if x2c <= x1c or y2c <= y1c:
        return None
    return frame[y1c:y2c, x1c:x2c]


@torch.no_grad()
def embed_crops(crops_bgr):
    out = []
    for i in tqdm(range(0, len(crops_bgr), EMBED_BATCH), desc="SigLIP"):
        pil = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops_bgr[i:i + EMBED_BATCH]]
        inputs = siglip_processor(images=pil, return_tensors="pt").to(DEVICE)
        e = siglip_model(**inputs).pooler_output.float().cpu().numpy()
        out.append(e / np.maximum(np.linalg.norm(e, axis=1, keepdims=True), 1e-12))
    return np.concatenate(out) if out else np.zeros((0, 768), np.float32)


print("SigLIP ready (torso crop, batched).")


## Cell 4 -- Choose the samples per id (metadata only, no video)
For each player id: its timeline is split into `SAMPLES_PER_ID` equal parts, and in each part the **largest isolated**
box is taken (no other box touching it -> no teammate/opponent inside the crop). Falls back to non-isolated boxes only
if an id has too few isolated ones.

In [ ]:
def _touch_matrix(B, e):
    w, h = (B[:, 2] - B[:, 0]) * e, (B[:, 3] - B[:, 1]) * e
    E = np.stack([B[:, 0] - w, B[:, 1] - h, B[:, 2] + w, B[:, 3] + h], 1)
    sep = ((E[:, None, 2] < E[None, :, 0]) | (E[None, :, 2] < E[:, None, 0]) |
           (E[:, None, 3] < E[None, :, 1]) | (E[None, :, 3] < E[:, None, 1]))
    t = ~sep
    np.fill_diagonal(t, False)
    return t

_player_set = set(player_ids)
cand = defaultdict(list)                        # tid -> [(frame, bbox, area, isolated)]
for f, d in tqdm(tracking_cache.items(), desc="Candidates", mininterval=1.0):
    if f % CANDIDATE_STRIDE or not d["tracks"]:
        continue
    B = np.array([[float(v) for v in t["bbox"]] for t in d["tracks"]])
    touch = _touch_matrix(B, ISO_EXPAND).any(1)
    for k, t in enumerate(d["tracks"]):
        if t["track_id"] in _player_set and not any(t.get(k) for k in SKIP_FOR_CROPS):
            a = (B[k, 2] - B[k, 0]) * (B[k, 3] - B[k, 1])
            if a >= MIN_BBOX_AREA:
                cand[t["track_id"]].append((f, t["bbox"], a, not touch[k]))

samples = {}                                     # tid -> [(frame, bbox)]
for tid in player_ids:
    c = cand[tid]
    iso = [x for x in c if x[3]]
    pool = iso if len(iso) >= SAMPLES_PER_ID else c
    if not pool:
        samples[tid] = []; continue
    pool.sort(key=lambda x: x[0])
    fr = np.array([x[0] for x in pool])
    edges = np.linspace(fr[0], fr[-1] + 1, SAMPLES_PER_ID + 1)
    pick = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        ks = np.where((fr >= lo) & (fr < hi))[0]
        if len(ks):
            k = ks[np.argmax([pool[i][2] for i in ks])]
            pick.append((pool[k][0], pool[k][1]))
    samples[tid] = pick

need = defaultdict(list)
for tid, lst in samples.items():
    for f, b in lst:
        need[f].append((tid, b))
print(f"Samples: {sum(len(v) for v in samples.values())} crops for {len(player_ids)} ids "
      f"(min per id {min((len(v) for v in samples.values()), default=0)}) from {len(need)} frames")


## Cell 5 -- One sequential video pass: torso crops of the samples, then SigLIP

In [ ]:
crop_list, crop_meta = [], []                    # meta: (tid, frame)
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"cannot open {VIDEO_PATH}"
last, f = max(need), 0
with tqdm(total=last + 1, desc="Reading video", unit="frame", mininterval=1.0) as pbar:
    while f <= last:
        if f in need:
            ok, frame = cap.read()
            if not ok: break
            for tid, bbox in need[f]:
                torso = get_torso_crop(frame, bbox)
                if torso is not None and torso.size:
                    crop_list.append(torso.copy()); crop_meta.append((tid, f))
        elif not cap.grab():
            break
        f += 1; pbar.update(1)
cap.release()

features = embed_crops(crop_list)
meta_tid = np.array([m[0] for m in crop_meta])
print(f"Embedded {len(features)} torso crops, feature shape {features.shape}")


## Cell 6 -- KMeans(k=2) on the crops, one vote per id, sanity sheet
Same scaler + KMeans as before. Each id's team = majority of its crops; `vote_share` = confidence.
The sheet shows random crops per cluster with their id: each row must be one kit.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
scaled = scaler.fit_transform(features)
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
crop_team = kmeans.fit_predict(scaled)
sil = silhouette_score(scaled, crop_team, sample_size=min(2000, len(scaled)), random_state=0)
print(f"Crop cluster sizes: {np.bincount(crop_team)} | silhouette: {sil:.3f}")

raw_team_votes, locked_team_by_id, vote_share_by_id = {}, {}, {}
for tid in player_ids:
    v = Counter(crop_team[meta_tid == tid].tolist())
    raw_team_votes[tid] = dict(v)
    if v:
        team, n = v.most_common(1)[0]
        locked_team_by_id[tid] = int(team)
        vote_share_by_id[tid] = round(n / sum(v.values()), 3)

print(f"\n{'id':>9} {'team':>4} {'share':>6} {'votes':>6}")
for tid in sorted(locked_team_by_id, key=lambda t: (locked_team_by_id[t], t)):
    flag = "  <-- weak" if vote_share_by_id[tid] < WEAK_MAJORITY_THRESHOLD else ""
    print(f"{show_id(tid):>9} {locked_team_by_id[tid]:>4} {vote_share_by_id[tid]:>6} "
          f"{sum(raw_team_votes[tid].values()):>6}{flag}")
print(f"\nPlayers per team: {Counter(locked_team_by_id.values())} (expected 10 / 10)")
_no = [t for t in player_ids if t not in locked_team_by_id]
if _no:
    print(f"  [warn] ids with no usable crop: {_no}")

N_SHOW = 12
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 1.3, 4))
for c in (0, 1):
    idx = np.where(crop_team == c)[0]
    for j, i in enumerate(rng.choice(idx, size=min(N_SHOW, len(idx)), replace=False)):
        axes[c][j].imshow(cv2.cvtColor(crop_list[i], cv2.COLOR_BGR2RGB))
        axes[c][j].set_title(f"{show_id(crop_meta[i][0])}\nf{crop_meta[i][1]}", fontsize=7)
    for ax in axes[c]:
        ax.axis("off")
plt.suptitle("each row = one cluster; single crops can be wrong, the per-id vote is what counts")
plt.tight_layout(); plt.show()


## Cell 7 -- Real kit colour per team (same torso crops, grass excluded, saturation boosted)

In [ ]:
def is_grass_dominant(px, thresh=GRASS_DOMINANCE_THRESH):
    b, g, r = px[:, 0], px[:, 1], px[:, 2]
    return (g > b * thresh) & (g > r * thresh)


def boost_saturation(bgr, factor=2.2, min_lightness=0.25, max_lightness=0.75):
    b, g, r = [c / 255.0 for c in bgr]
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    r, g, b = colorsys.hls_to_rgb(h, max(min_lightness, min(max_lightness, l)), min(1.0, s * factor))
    return (int(b * 255), int(g * 255), int(r * 255))


sums = {0: np.zeros(3), 1: np.zeros(3)}; counts = {0: 0, 1: 0}
for crop, (tid, f) in zip(crop_list, crop_meta):
    team = locked_team_by_id.get(tid)            # colour by the id's team, not the single-crop cluster
    if team is None:
        continue
    px = crop.reshape(-1, 3).astype(np.float64)
    kept = px[~is_grass_dominant(px)]
    sums[team] += kept.sum(axis=0); counts[team] += len(kept)
team_colors_raw = {t: tuple(int(v) for v in (sums[t] / counts[t])) if counts[t] else (128, 128, 128) for t in (0, 1)}
team_colors = {t: boost_saturation(c) for t, c in team_colors_raw.items()}
print(f"Raw kit colours (BGR): {team_colors_raw}\nBoosted (saved):       {team_colors}")

fig, axes = plt.subplots(1, 2, figsize=(4, 2))
for t in (0, 1):
    axes[t].imshow(np.full((10, 10, 3), team_colors[t][::-1], np.uint8)); axes[t].set_title(f"team {t}"); axes[t].axis("off")
plt.show()


## Cell 8 -- Goalkeepers: nearest team centroid in pitch space (same rule as before)

In [ ]:
def project_to_pitch(point_px, H):
    pt = cv2.perspectiveTransform(np.array([[[point_px[0], point_px[1]]]], dtype=np.float32), np.asarray(H)).reshape(2)
    return float(pt[0] / PX_PER_METER), float(pt[1] / PX_PER_METER)


pos_by_frame = defaultdict(dict)                 # frame -> {tid: (x, y) m}
_want = set(locked_team_by_id) | set(goalkeeper_ids)
for f, d in tracking_cache.items():
    if f % POSITION_SAMPLE_STRIDE:
        continue
    H = get_homography(f)
    if H is None:
        continue
    for t in d["tracks"]:
        if t["track_id"] in _want and not t.get("duplicate"):
            x1, y1, x2, y2 = t["bbox"]
            pos_by_frame[f][t["track_id"]] = project_to_pitch(((x1 + x2) / 2, y2), H)

team_positions = {0: [], 1: []}
for f, P in pos_by_frame.items():
    for tid, p in P.items():
        if tid in locked_team_by_id:
            team_positions[locked_team_by_id[tid]].append(p)
team_centroids = {t: (np.mean(p, axis=0) if p else None) for t, p in team_positions.items()}
print(f"Team 0 centroid (pitch m): {team_centroids[0]}\nTeam 1 centroid (pitch m): {team_centroids[1]}")

# each sampled frame votes: the goalkeeper belongs to the team whose players are closest to him IN THAT FRAME
# (a whole-match centroid fails when the teams swap sides at half time)
goalkeeper_team_assignment, gk_vote_share = {}, {}
for gk in goalkeeper_ids:
    votes = Counter()
    for f, P in pos_by_frame.items():
        if gk not in P:
            continue
        pts = {k: [p for tid, p in P.items() if locked_team_by_id.get(tid) == k] for k in (0, 1)}
        if min(len(pts[0]), len(pts[1])) < GK_MIN_PLAYERS_PER_TEAM:
            continue
        g = np.asarray(P[gk])
        d = {k: float(np.linalg.norm(g - np.mean(pts[k], axis=0))) for k in (0, 1)}
        votes[0 if d[0] < d[1] else 1] += 1
    if not votes:
        print(f"  GK {show_id(gk)}: no frame with both teams on screen -> not assigned"); continue
    team, n = votes.most_common(1)[0]
    goalkeeper_team_assignment[gk] = team
    gk_vote_share[gk] = round(n / sum(votes.values()), 3)
    flag = "  <-- weak" if gk_vote_share[gk] < WEAK_MAJORITY_THRESHOLD else ""
    print(f"  GK {show_id(gk)}: votes {dict(votes)} -> team {team} (share {gk_vote_share[gk]}){flag}")
if len(set(goalkeeper_team_assignment.values())) < len(goalkeeper_team_assignment):
    print("  [warn] both goalkeepers got the same team -> check the GK ids / homography")


## Cell 9 -- Save both structures (normal + simple)

In [ ]:
final_team_by_id = dict(locked_team_by_id)
final_team_by_id.update(goalkeeper_team_assignment)
track_team_segments = {tid: [(0, t)] for tid, t in final_team_by_id.items()}   # one segment: ids are stable
switch_suspects = []

with open(OUTPUT_TEAM_CACHE_PATH, "wb") as f:                                   # normal structure (old keys)
    pickle.dump({
        "team_by_id": final_team_by_id,
        "track_team_segments": track_team_segments,
        "switch_suspects": switch_suspects,
        "raw_team_votes": raw_team_votes,
        "goalkeeper_team_assignment": goalkeeper_team_assignment,
        "team_centroids": team_centroids,
        "team_colors": team_colors,
    }, f)

vote_share_by_id.update(gk_vote_share)
team_by_id = {tid: final_team_by_id.get(tid) for tid in sorted(locked_class_by_id)}   # referees / others -> None
with open(OUTPUT_TEAM_SIMPLE_PATH, "wb") as f:                                  # simple structure
    pickle.dump({"team_by_id": team_by_id, "vote_share_by_id": vote_share_by_id, "team_colors": team_colors}, f)

print(f"{'id':>9} {'class':<11} {'team':>5} {'share':>6}")
for tid, t in team_by_id.items():
    if locked_class_by_id[tid] not in ("player", "goalkeeper", "referee"):
        continue
    print(f"{show_id(tid):>9} {locked_class_by_id[tid]:<11} {str(t):>5} {str(vote_share_by_id.get(tid, '-')):>6}")
print(f"\nTeam sizes (players + GK): {Counter(t for t in team_by_id.values() if t is not None)}")
print(f"Saved normal -> {OUTPUT_TEAM_CACHE_PATH}\nSaved simple -> {OUTPUT_TEAM_SIMPLE_PATH}")


## Cell 10 -- (optional) Annotated check video, `RENDER_VIDEO = True`

In [ ]:
GOALKEEPER_COLOR, REFEREE_COLOR = (0, 165, 255), (40, 40, 230)
BALL_COLOR, BALL_LOW_CONF_COLOR = (0, 220, 255), (0, 140, 200)


def draw_pill_label(frame, text, x, y, color):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(text, font, 0.5, 1)
    x1, y1, x2, y2 = int(x), int(y - th - 8), int(x + tw + 12), int(y)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, -1, cv2.LINE_AA)
    cv2.putText(frame, text, (x1 + 6, y2 - 4), font, 0.5, (255, 255, 255), 1, cv2.LINE_AA)


if RENDER_VIDEO:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    fi = 0
    with tqdm(total=total, desc="Rendering", unit="frame", mininterval=1.0) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            data = tracking_cache.get(fi, {"ball": None, "tracks": []})
            for t in data["tracks"]:
                tid = t["track_id"]; role = locked_class_by_id.get(tid, t["class"])
                x1, y1, x2, y2 = [int(v) for v in t["bbox"]]
                num = player_info.get(tid, {}).get("number")
                txt = str(num) if num is not None else f"#{tid}"
                if role == "referee":
                    color, label = REFEREE_COLOR, "REF"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
                elif role == "goalkeeper":
                    color, label = GOALKEEPER_COLOR, f"GK {txt} T{final_team_by_id.get(tid, '?')}"
                    cv2.ellipse(frame, ((x1 + x2) // 2, y2), (int((x2 - x1) * 0.5), 8), 0, 0, 360, color, 3, cv2.LINE_AA)
                else:
                    team = final_team_by_id.get(tid, -1)
                    color, label = team_colors.get(team, (200, 200, 200)), f"{txt} T{team}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
                draw_pill_label(frame, label, x1, y1, color)
            if data.get("ball") is not None:
                bx1, by1, bx2, by2 = [int(v) for v in data["ball"]["bbox"]]
                low = data["ball"].get("low_confidence", False)
                cv2.circle(frame, ((bx1 + bx2) // 2, (by1 + by2) // 2), max((bx2 - bx1) // 2, 4),
                           BALL_LOW_CONF_COLOR if low else BALL_COLOR, 1 if low else 2, cv2.LINE_AA)
            writer.write(frame); fi += 1; pbar.update(1)
    cap.release(); writer.release()
    print(f"Saved {OUTPUT_VIDEO_PATH}")
else:
    print("RENDER_VIDEO=False -> skipped")
